# 原 Stage1 的 NVDA 指引实现状态输入预检

按 `source_preflight_plan.json` 只读既有Qlib Recorder，验证原Stage1 2011–2022训练日历和2023+完整账户日历的来源身份、09:29决策时点及已知状态。0拟合、0账户、0市场收益读取。

In [1]:
import torch
from pathlib import Path
import hashlib,json
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.nvda_own_guidance_realization import align_realizations,FEATURE,FLAGS
root=Path.cwd();card=root/'research/stage1_nvda_guidance_receiver_20261022'
plan=json.loads((card/'source_preflight_plan.json').read_text())
objects={};summaries={}
for name,source in plan['sources'].items():
    path=root/source['card'];s=json.loads((path/'summary.json').read_text())
    assert s['recorder_id']==source['recorder_id']
    manifest=json.loads((path/'verification_manifest.json').read_text())
    hits=[x for x in manifest['entries'] if x['path'].endswith('/artifacts/'+source['artifact'])]
    assert len(hits)==1 and hits[0]['sha256']==source['sha256']
    assert hashlib.sha256((path/hits[0]['path']).read_bytes()).hexdigest()==source['sha256']
    summaries[name]=s
qlib.init(provider_uri=summaries['training_timing']['provider_uri'],region='us')
for name,source in plan['sources'].items():
    s=summaries[name]
    with R.uri_context(s['recorder_uri']):
        recorder=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id'])
        objects[name]=recorder.load_object(source['artifact'])
print('Three source SHA identities and Recorder reads passed.')

Three source SHA identities and Recorder reads passed.


In [2]:
timing=objects['training_timing'];frame=objects['evaluation_calendar'];pairs=objects['guidance_pairs']
train_timing=timing.loc[timing.index<pd.Timestamp('2023-01-01'),'decision_at']
assert len(train_timing)==3020 and len(frame)==944
eval_index=pd.DatetimeIndex(frame.signal_date)
eval_trade=pd.to_datetime(frame.trade_date).dt.tz_localize('America/New_York')
eval_timing=pd.Series(eval_trade.array+pd.Timedelta(hours=9,minutes=29),index=eval_index)
assert eval_timing.dt.hour.eq(9).all() and eval_timing.dt.minute.eq(29).all()
assert (eval_timing.reset_index(drop=True)<pd.to_datetime(frame.decision_at).dt.tz_localize('America/New_York')).all()
train_features,train_trace,events=align_realizations(pairs,train_timing)
eval_features,eval_trace,_=align_realizations(pairs,eval_timing)
for features,trace in [(train_features,train_trace),(eval_features,eval_trace)]:
    assert features.loc[features[FLAGS[1]].eq(0),FEATURE].isna().all()
    assert features[FLAGS].isin([0.,1.]).all().all()
    valid=trace.source_available_at.notna()
    assert (pd.to_datetime(trace.loc[valid,'source_available_at'],utc=True)<=pd.to_datetime(trace.loc[valid,'decision_at'],utc=True)).all()
def coverage(x):
    return {'rows':len(x),'seen':int(x[FLAGS[0]].sum()),'comparable':int(x[FLAGS[1]].sum()),
            'known_preannounced':int(x[FLAGS[2]].sum()),'finite_numeric':int(x[FEATURE].notna().sum())}
result={'status':'conditional_input_coverage_verified_not_PIT','plan_sha256':hashlib.sha256((card/'source_preflight_plan.json').read_bytes()).hexdigest(),
        'source_recorder_ids':{key:value['recorder_id'] for key,value in summaries.items()},
        'training_calendar':coverage(train_features),'evaluation_calendar':coverage(eval_features),
        'train_first_decision':str(train_timing.min()),'train_last_decision':str(train_timing.max()),
        'evaluation_first_decision':str(eval_timing.min()),'evaluation_last_decision':str(eval_timing.max()),
        'source_realized_report_events':len(events),'historical_first_public_timestamp_verified':False,
        'all_updates_coverage_verified':False,'original_html_bytes_verified':False,
        'new_model_fits':0,'new_native_accounts':0,'market_outcome_reads':0,
        'interpretation':'Coverage and delayed alignment permit only a historical development design. Exact source/publication and future economic gates remain open.'}
(card/'source_preflight_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False,indent=2))

{
  "status": "conditional_input_coverage_verified_not_PIT",
  "plan_sha256": "7012c9a1e2671feb29f19fa3ebdefd1c5bc99ee5fdfeb2adcbe58c91afa02b40",
  "source_recorder_ids": {
    "training_timing": "68e31a0a0fad44e7ba1d1f7d856c7a0d",
    "evaluation_calendar": "6e227278b7a64eafbcf34b965e16d2b9",
    "guidance_pairs": "9fb35ff59c1b43b7ba3906a46d7002c0"
  },
  "training_calendar": {
    "rows": 3020,
    "seen": 2929,
    "comparable": 2865,
    "known_preannounced": 185,
    "finite_numeric": 2865
  },
  "evaluation_calendar": {
    "rows": 944,
    "seen": 944,
    "comparable": 944,
    "known_preannounced": 0,
    "finite_numeric": 944
  },
  "train_first_decision": "2011-01-04 09:29:00-05:00",
  "train_last_decision": "2023-01-03 09:29:00-05:00",
  "evaluation_first_decision": "2023-01-04 09:29:00-05:00",
  "evaluation_last_decision": "2026-10-08 09:29:00-04:00",
  "source_realized_report_events": 62,
  "historical_first_public_timestamp_verified": false,
  "all_updates_coverage_verif

In [3]:
from dataclasses import asdict
from src.model.stage1_nvda_guidance import Stage1NvdaGuidanceAllocator,initialization_contract
from src.workflow.pit_constituent_auxiliary_training_allocation import COMMON_FEATURES
implementation=json.loads((card/'implementation_preflight_plan.json').read_text())
assert hashlib.sha256((card/'source_preflight_plan.json').read_bytes()).hexdigest()==implementation['source_preflight_plan_sha256']
assert hashlib.sha256((root/implementation['model_adapter']).read_bytes()).hexdigest()==implementation['model_adapter_sha256']
source_card=root/implementation['training_source_card']
manifest=json.loads((source_card/'verification_manifest.json').read_text())
source=summaries['training_timing']
for name,digest in implementation['additional_training_artifacts_sha256'].items():
    entries=[e for e in manifest['entries'] if e['path'].endswith('/artifacts/'+name)]
    assert len(entries)==1 and entries[0]['sha256']==digest
    assert hashlib.sha256((source_card/entries[0]['path']).read_bytes()).hexdigest()==digest
with R.uri_context(source['recorder_uri']):
    recorder=R.get_recorder(recorder_id=source['recorder_id'],experiment_id=source['experiment_id'])
    base_features=recorder.load_object('inputs/features')
    original_models={seed:recorder.load_object('models/base_outer_'+str(seed)) for seed in implementation['expected']['seeds']}
full_guidance,full_trace,_=align_realizations(pairs,timing.decision_at)
pd.testing.assert_index_equal(base_features.index,full_guidance.index,check_names=False)
assert not set([*FLAGS,FEATURE]).intersection(base_features.columns)
joined=base_features.join(full_guidance)
train=joined.loc[joined.index<pd.Timestamp('2023-01-01')]
assert len(train)==implementation['expected']['pre2023_calendar_rows']
assert int(train[FLAGS[1]].sum())==implementation['expected']['guidance_comparable_calendar_rows']
assert len(COMMON_FEATURES)==implementation['expected']['auxiliary_common_features']
parity=[]
for seed,original in original_models.items():
    assert not original.extra_features and len(original.base_features)==implementation['expected']['base_primary_features']
    args=dict(base_features=original.base_features,spec={**asdict(original.spec),'root':str(root)},
              seed=original.seed,cutoff=str(original.cutoff),train_end=str(original.train_end.date()),
              minimum_observations=original.minimum_observations)
    b=Stage1NvdaGuidanceAllocator(**args,mode='neutral')
    c=Stage1NvdaGuidanceAllocator(**args,mode='observed')
    proof=initialization_contract(original,b,c)
    b._fit_extra_scaler(train);c._fit_extra_scaler(train)
    assert b.extra_scaler==c.extra_scaler
    left,right=b._extra_tensor(train),c._extra_tensor(train)
    assert torch.equal(left[:,:len(FLAGS)],right[:,:len(FLAGS)]) and left[:,-1].eq(0).all()
    assert int(right[:,-1].ne(0).sum())>0
    parity.append({'seed':seed,**proof,'calendar_scaler_rows':b.extra_scaler[FEATURE]['observations'],
                   'candidate_nonzero_numeric_rows':int(right[:,-1].ne(0).sum())})
result2={'status':'no_fit_adapter_and_source_mechanics_passed',
         'plan_sha256':hashlib.sha256((card/'implementation_preflight_plan.json').read_bytes()).hexdigest(),
         'source_recorder_id':source['recorder_id'],'seeds':parity,'base_primary_features':39,
         'common_features':len(COMMON_FEATURES),'train_calendar_rows':len(train),
         'train_calendar_comparable':int(train[FLAGS[1]].sum()),
         'model_fits':0,'native_accounts':0,'market_outcome_reads':0,
         'limitation':'Calendar-only scaler preview is not the formal mature-label training scaler. This preflight does not authorize fitting or account execution.'}
(card/'implementation_preflight_result.json').write_text(json.dumps(result2,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result2,ensure_ascii=False,indent=2))

{
  "status": "no_fit_adapter_and_source_mechanics_passed",
  "plan_sha256": "da15412b4e4a89344b577ba3b01fae7687b0515e8c64d9d89a597222c30f92ce",
  "source_recorder_id": "68e31a0a0fad44e7ba1d1f7d856c7a0d",
  "seeds": [
    {
      "seed": 71,
      "original_tensors_equal_after_projection": true,
      "paired_initial_tensors_equal": true,
      "all_initial_outputs_equal": true,
      "added_weights_zero": true,
      "calendar_scaler_rows": 2865,
      "candidate_nonzero_numeric_rows": 2865
    },
    {
      "seed": 172,
      "original_tensors_equal_after_projection": true,
      "paired_initial_tensors_equal": true,
      "all_initial_outputs_equal": true,
      "added_weights_zero": true,
      "calendar_scaler_rows": 2865,
      "candidate_nonzero_numeric_rows": 2865
    },
    {
      "seed": 273,
      "original_tensors_equal_after_projection": true,
      "paired_initial_tensors_equal": true,
      "all_initial_outputs_equal": true,
      "added_weights_zero": true,
      "cal

In [4]:
binding_plan=json.loads((card/'calendar_binding_plan.json').read_text())
bound={};source_summaries={}
for role,source in binding_plan['sources'].items():
    path=root/source['card'];summary=json.loads((path/'summary.json').read_text())
    assert summary['recorder_id']==source['recorder_id']
    manifest=json.loads((path/'verification_manifest.json').read_text())
    hits=[x for x in manifest['entries'] if x['path'].endswith('/artifacts/'+source['artifact'])]
    assert len(hits)==1 and hits[0]['sha256']==source['sha256']
    assert hashlib.sha256((path/hits[0]['path']).read_bytes()).hexdigest()==source['sha256']
    source_summaries[role]=summary
for role,source in binding_plan['sources'].items():
    summary=source_summaries[role]
    with R.uri_context(summary['recorder_uri']):
        recorder=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
        bound[role]=recorder.load_object(source['artifact'])
model_predictions=bound['training_predictions'];funded=bound['Stage1_frame'][['signal_date','trade_date','exit_date']].reset_index(drop=True)
account=bound['Stage1_account']['replay'][['signal_date','trade_date','exit_date']].reset_index(drop=True)
pd.testing.assert_frame_equal(funded,account)
assert len(funded)==binding_plan['expected']['funded_intervals']==944
assert funded.signal_date.iloc[0].strftime('%Y-%m-%d')==binding_plan['expected']['first_signal']
assert funded.trade_date.iloc[0].strftime('%Y-%m-%d')==binding_plan['expected']['first_trade']
assert funded.signal_date.iloc[-1].strftime('%Y-%m-%d')==binding_plan['expected']['last_signal']
assert funded.trade_date.iloc[-1].strftime('%Y-%m-%d')==binding_plan['expected']['last_trade']
assert funded.exit_date.iloc[-1].strftime('%Y-%m-%d')==binding_plan['expected']['terminal_exit']
assert (funded.signal_date<funded.trade_date).all() and (funded.trade_date<funded.exit_date).all()
assert funded.exit_date.iloc[:-1].reset_index(drop=True).equals(funded.trade_date.iloc[1:].reset_index(drop=True))
assert pd.DatetimeIndex(eval_timing.index).equals(pd.DatetimeIndex(funded.signal_date))
assert pd.DatetimeIndex(eval_timing.dt.tz_localize(None).dt.normalize()).equals(pd.DatetimeIndex(funded.trade_date))
saved_funded_clock=bound['saved_training_timing'].loc[pd.DatetimeIndex(funded.signal_date),'decision_at']
pd.testing.assert_series_equal(saved_funded_clock,eval_timing,check_names=False)
assert str(saved_funded_clock.iloc[0])=='2023-01-04 09:29:00-05:00'
funded_signals=pd.DatetimeIndex(funded.signal_date)
for seed in binding_plan['expected'].get('seeds',[71,172,273]):
    predict=model_predictions[('base',seed)]
    assert len(predict)==binding_plan['expected']['all_model_prediction_rows']==946
    pd.testing.assert_index_equal(predict.index[:944],funded_signals,check_names=False)
    tail=[d.strftime('%Y-%m-%d') for d in predict.index[944:]]
    assert tail==binding_plan['expected']['unfunded_tail_predictions']
serial='\n'.join('|'.join(str(v.date()) for v in row) for row in funded[['signal_date','trade_date','exit_date']].itertuples(index=False,name=None))+'\n'
result3={'status':'exact_944_signal_trade_exit_binding_verified','plan_sha256':hashlib.sha256((card/'calendar_binding_plan.json').read_bytes()).hexdigest(),
         'funded_date_tuple_sha256':hashlib.sha256(serial.encode()).hexdigest(),
         'first_tuple':[str(x.date()) for x in funded.iloc[0]],'last_tuple':[str(x.date()) for x in funded.iloc[-1]],
         'prediction_rows_per_seed':946,'funded_prediction_rows':944,'unfunded_prediction_tail':tail,
         'native_account_calendar_matches':True,'conservative_0929_decision_matches_trade_date':True,
         'saved_training_timing_matches_all_944_trade_date_0929_clocks':True,
         'first_signal_saved_decision_at':str(saved_funded_clock.iloc[0]),
         'model_fits':0,'new_accounts':0,'return_values_used':0}
(card/'calendar_binding_result.json').write_text(json.dumps(result3,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result3,ensure_ascii=False,indent=2))

{
  "status": "exact_944_signal_trade_exit_binding_verified",
  "plan_sha256": "43df1af98eecb7dd128279a6d3d7eaca8e364b79a3cd1bb48b6c1cb9ea188205",
  "funded_date_tuple_sha256": "8a268b6e1af6d8153bf743c470916f8283efc89fb610d9ee8b6c679e3924db6a",
  "first_tuple": [
    "2023-01-03",
    "2023-01-04",
    "2023-01-05"
  ],
  "last_tuple": [
    "2026-10-07",
    "2026-10-08",
    "2026-10-09"
  ],
  "prediction_rows_per_seed": 946,
  "funded_prediction_rows": 944,
  "unfunded_prediction_tail": [
    "2026-10-08",
    "2026-10-09"
  ],
  "native_account_calendar_matches": true,
  "conservative_0929_decision_matches_trade_date": true,
  "saved_training_timing_matches_all_944_trade_date_0929_clocks": true,
  "first_signal_saved_decision_at": "2023-01-04 09:29:00-05:00",
  "model_fits": 0,
  "new_accounts": 0,
  "return_values_used": 0
}


## 中文结论

已从Qlib Recorder核对保守09:29来源时点：训练日历3020日中可比2865日，944个评价信号日均可比。三seed原Stage1与匹配B/C的初始输出一致，B只见来源标记、C另见标准化实现值。逐行日历绑定证明2023-01-03信号在2023-01-04决策/成交，原预测946行只有944行被资金账户使用。训练Dataset的独立无拟合预检又核对3014条成熟主样本、20741条辅助样本、2859条可比主样本；见同卡`training_dataset_preflight.json`。

训练及日历阶段：dataset_and_calendar_bound_design_only_no_fit。0优化器步、0新原生账户，尚无新增信息或净SOXX结果。独立设计复核要求的原价账本/控制器与B/C状态隔离仍须完成并复核；历史首次公开、更新完整性及原HTML字节仍未认证。

原价账户另以`baseline_parity_plan.json`完成一次Stage1 A 0bp参考复现：944行、1275笔订单、3778估值点与原生来源一致，NAV/现金/收益最大差0，且scikit-learn版本为源模型保存时的1.9.1。该步骤未执行B/C拟合或账户，当前卡片决定为native_account_source_and_A_parity_checked_no_fit。

## 冻结 0bp 配对执行入口

独立设计复核和与当前计划、代码哈希绑定的执行许可齐全后，按 B/C 各三 seed 完整拟合，再封存两个目标，最后做两本原价 Qlib 独立账户。执行会更新本卡的 `summary.json`、`conclusion.md` 和 Recorder；若守卫失败，停在相应阶段并保留尝试记录。


In [5]:
from src.workflow.stage1_nvda_guidance_receiver import run
result = run(root=root, card=card)
{key: result[key] for key in ("decision", "historical_0bp_candidate", "new_fits", "new_native_accounts", "recorder_id")}


{'decision': 'not_promoted_fixed_Stage1_guidance_receiver_0bp',
 'historical_0bp_candidate': None,
 'new_fits': 6,
 'new_native_accounts': 2,
 'recorder_id': 'f284f6d663b647b297c84d62c7bffa85'}

## 固定失败后的区间归因

只读已完成的 Qlib 原生账户，对照原 Stage1 A、B、C 与 SOXX 的相同资金区间。季度和最差连续126日用于解释失败，不用于选取交易窗口或修改此固定版本。


In [6]:
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources
import numpy as np
import pandas as pd
from qlib.workflow import R
bound = prepare_account_sources(root=root, card=card)
summary = json.loads((card/'summary.json').read_text())
with R.uri_context(summary['recorder_uri']):
    rec = R.get_recorder(recorder_id=summary['recorder_id'], experiment_id=summary['experiment_id'])
    arms = {name: rec.load_object('native/'+name+'_0')['authority'] for name in ('B_flags_only','C_realization')}
soxx = bound['references']['SOXX']['authority']
a = bound['references']['Stage1_A']['authority']
assert all(arms[name][['signal_date','trade_date','exit_date']].equals(soxx[['signal_date','trade_date','exit_date']]) for name in arms)
selected = ['2023Q4','2025Q2','2025Q3','2026Q2','2026Q3']
labels = pd.to_datetime(soxx.trade_date).dt.to_period('Q').astype(str)
rows = []
for quarter in selected:
    take = labels.eq(quarter)
    for name, book in [('A_Stage1',a),*arms.items()]:
        own = np.log1p(book.loc[take,'strategy_return'].to_numpy(float))
        reference = np.log1p(soxx.loc[take,'strategy_return'].to_numpy(float))
        rows.append({'quarter':quarter,'family':name,'sessions':int(take.sum()),
            'relative_log_soxx':float((own-reference).sum()),
            'relative_wealth_soxx':float(np.exp((own-reference).sum())),
            'mean_ending_cash_weight':float(book.loc[take,'ending_cash_weight'].mean()),
            'mean_ending_soxl_weight':float(book.loc[take,'ending_soxl_weight'].mean()),
            'mean_ending_soxx_weight':float(book.loc[take,'ending_soxx_weight'].mean()),
            'execute_open_fraction':float(book.loc[take,'execute_open_target'].astype(float).mean()),
            'mean_monitored_drawdown':float(book.loc[take,'monitored_drawdown'].mean())})
for name,book in arms.items():
    logs=np.log1p(book.strategy_return.to_numpy(float))-np.log1p(soxx.strategy_return.to_numpy(float))
    low=int(pd.Series(logs).rolling(126).sum().iloc[125:].argmin())
    rows.append({'quarter':'worst_contiguous_126','family':name,'sessions':126,
        'first_trade':str(soxx.trade_date.iloc[low].date()),
        'last_exit':str(soxx.exit_date.iloc[low+125].date()),
        'relative_log_soxx':float(logs[low:low+126].sum()),
        'relative_wealth_soxx':float(np.exp(logs[low:low+126].sum()))})
report={'status':'read_only_native_qlib_regime_attribution','source_recorder_id':summary['recorder_id'],
    'reference_stage1_recorder_id':plan['parent']['account_recorder_id'] if 'parent' in plan else json.loads((card/'plan.json').read_text())['parent']['account_recorder_id'],
    'source_native_accounts':['B_flags_only_0','C_realization_0'],
    'periods':selected,'rows':rows,'new_fits':0,'new_accounts':0,
    'regimes_are_descriptive_not_trading_rules':True,'causal_claim':False}
(card/'regime_attribution.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
pd.DataFrame(rows)


                 quarter         family  sessions  relative_log_soxx  \
0                 2023Q4       A_Stage1        63          -0.109349   
1                 2023Q4   B_flags_only        63          -0.071825   
2                 2023Q4  C_realization        63           0.015601   
3                 2025Q2       A_Stage1        62          -0.062982   
4                 2025Q2   B_flags_only        62          -0.159675   
5                 2025Q2  C_realization        62          -0.160820   
6                 2025Q3       A_Stage1        64          -0.013206   
7                 2025Q3   B_flags_only        64          -0.062027   
8                 2025Q3  C_realization        64          -0.153985   
9                 2026Q2       A_Stage1        62           0.148844   
10                2026Q2   B_flags_only        62           0.052928   
11                2026Q2  C_realization        62          -0.256144   
12                2026Q3       A_Stage1        64          -0.16

## 固定 0bp 最终结论

原 Stage1 接收层在相同 Qlib 原价资金日历下完成 B/C 各三 seed、两本独立 0bp 账户。B 对 SOXX 的期末财富比 0.743486、C 为 0.491631；B 的 819 个 126 日窗口只有 30.28% 正超额，C 为 37.97%，两者 SOXX 配对区块下界都为负。C 相对 B 期末财富比 0.661251，数值实现值未获增量支持。B 在 2025Q2 和 2026Q3 偏现金，C 在 2026Q2 进一步落后；均属描述性归因，不能当作调仓规则。

决定：`not_promoted_fixed_Stage1_guidance_receiver_0bp`。本版关闭，不按事后季度调来源延迟、数值符号、标准化或风险阈值。2023–2026 历史已重复使用，首次公开时点、全分钟回撤和独立前向仍未合格。完整门、区间、Recorder 与 NAS 备份见 `summary.json`、`regime_attribution.json`、`conclusion.md`。下一步先核对实质不同的信息来源的训练覆盖和前向发布时间。